# Pipeline Sweep Synthesis
Quantized LoRA fine-tuning (`train.py`) and an Azure ML hyperparameter sweep (`submit_sweep.py`).

## Setup
Run the install once, then **restart the kernel** (Kernel → Restart Kernel) before running the cells below.

In [ ]:
%pip install torch transformers peft mlflow bitsandbytes azure-ai-ml azure-identity

## Part 1: The Training Script (train.py)
### Task 1: Configure 4-bit Quantization

In [ ]:
import torch
from transformers import BitsAndBytesConfig

# Define BitsAndBytesConfig for 4-bit NF4 quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
print(quant_config)

### Task 2: Configure LoRA Adapters
Tested against a mock `args` object, because `get_args()` (argparse) fails inside a notebook kernel.

In [ ]:
from peft import LoraConfig, TaskType

class Args:
    rank = 8
    alpha = 16

args = Args()

# Configure LoraConfig targeting attention projection layers
peft_config = LoraConfig(
    r=args.rank,
    lora_alpha=args.alpha,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    task_type=TaskType.CAUSAL_LM,
)
print(peft_config)

### Task 3: Implement Rank-0 Protected Logging

In [ ]:
import os
import mlflow

for epoch in range(5):
    # Mocked loss calculation
    loss = torch.tensor(0.85, requires_grad=True) * 0.8

    # Rank-0 protected MLflow tracking
    if os.environ.get("RANK", "0") == "0":
        mlflow.log_metric("train_loss", loss.item(), step=epoch)
        print(f"epoch {epoch}: logged train_loss={loss.item():.4f}")

### Export train.py

In [ ]:
%%writefile train.py
import os
import argparse
import torch
import mlflow
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType


def get_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--learning_rate", type=float, default=2e-4)
    parser.add_argument("--rank", type=int, default=8)
    parser.add_argument("--alpha", type=int, default=16)
    return parser.parse_args()


def main():
    args = get_args()

    # Task 1: load the base model in 4-bit NormalFloat (NF4) with double quantization.
    # Weights are stored in 4-bit; matrix multiplications are upcast to float16.
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # Task 2: LoRA adapters on the self-attention projections.
    # r and lora_alpha come from argparse so each sweep trial gets its own values.
    peft_config = LoraConfig(
        r=args.rank,
        lora_alpha=args.alpha,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        task_type=TaskType.CAUSAL_LM,
    )

    for epoch in range(5):
        loss = torch.tensor(0.85, requires_grad=True) * 0.8

        # Task 3: rank-0 protected MLflow tracking.
        # Only the global rank-0 process writes to MLflow; RANK defaults to "0" for local runs.
        if os.environ.get("RANK", "0") == "0":
            mlflow.log_metric("train_loss", loss.item(), step=epoch)


if __name__ == "__main__":
    main()


## Part 2: The Sweep Submission Script (submit_sweep.py)
### Task 4: Define the Parameter Search Space

In [ ]:
from azure.ai.ml.sweep import Choice, Uniform

# Define the search space (keys match the ${{inputs.*}} names in the command string)
search_space = {
    "learning_rate": Uniform(min_value=1e-5, max_value=1e-4),
    "rank": Choice(values=[8, 16, 32]),
}
print(search_space)

### Task 5: Configure the Early Stopping Policy and Sweep Job
The job is built and printed only; it is not submitted.

In [ ]:
from azure.ai.ml import command
from azure.ai.ml.sweep import BanditPolicy

# Define the Bandit early stopping policy
early_termination_policy = BanditPolicy(
    slack_factor=0.1,
    evaluation_interval=2,
    delay_evaluation=5,
)
print(early_termination_policy)

base_run = command(
    code="./src",
    command="python train.py --learning_rate ${{inputs.learning_rate}} --rank ${{inputs.rank}}",
    inputs={"learning_rate": 2e-4, "rank": 8},
    compute="gpu-cluster",
)

sweep_job = base_run.sweep(
    search_space=search_space,
    sampling_algorithm="random",
    primary_metric="train_loss",
    goal="Minimize",
    early_termination_policy=early_termination_policy,
)
sweep_job.set_limits(max_total_trials=20, max_concurrent_trials=4, timeout=7200)
print(sweep_job)

### Export submit_sweep.py

In [ ]:
%%writefile submit_sweep.py
from azure.ai.ml import MLClient, command
from azure.ai.ml.sweep import Choice, Uniform, BanditPolicy
from azure.identity import DefaultAzureCredential


def submit_sweep():
    ml_client = MLClient(DefaultAzureCredential(), "sub_id", "rg_name", "ws_name")

    base_run = command(
        code="./src",  # folder that holds train.py when the job is submitted
        command="python train.py --learning_rate ${{inputs.learning_rate}} --rank ${{inputs.rank}}",
        inputs={"learning_rate": 2e-4, "rank": 8},  # defaults; the sweep overrides them per trial
        compute="gpu-cluster",
    )

    # Task 4: search space. Keys must match the ${{inputs.*}} names in the command string.
    search_space = {
        "learning_rate": Uniform(min_value=1e-5, max_value=1e-4),
        "rank": Choice(values=[8, 16, 32]),
    }

    # Task 5: Bandit early termination. Cancel any trial whose train_loss is more than
    # 10% worse than the best trial, checked every 2 intervals after the first 5.
    early_termination_policy = BanditPolicy(
        slack_factor=0.1,
        evaluation_interval=2,
        delay_evaluation=5,
    )

    sweep_job = base_run.sweep(
        search_space=search_space,
        sampling_algorithm="random",
        primary_metric="train_loss",  # must match the metric name logged in train.py
        goal="Minimize",
        early_termination_policy=early_termination_policy,
    )

    # Hard budget caps on the sweep as a whole.
    sweep_job.set_limits(max_total_trials=20, max_concurrent_trials=4, timeout=7200)

    print(sweep_job)  # prints the YAML Azure ML would receive
    # ml_client.jobs.create_or_update(sweep_job)  # Not required for this activity


if __name__ == "__main__":
    submit_sweep()


### Confirm both files exist

In [ ]:
import os
for f in ["train.py", "submit_sweep.py"]:
    print(f, "exists:", os.path.exists(f))